# 🌱 Exploratory Data Analysis - Crop Recommendation System
## Enterprise Agricultural Intelligence Platform

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# Set style
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette("husl")

# Load data
df = pd.read_csv('../data/datasets/crop_data.csv')
print(f"Dataset shape: {df.shape}")
print(f"Columns: {list(df.columns)}")
print(f"\nFirst 5 rows:")
df.head()

In [ ]:
# Basic statistics
df.describe()

In [ ]:
# Crop distribution
crop_counts = df['crop'].value_counts()

fig = px.bar(
    x=crop_counts.values,
    y=crop_counts.index,
    orientation='h',
    title='Distribution of Crops in Dataset',
    labels={'x': 'Number of Samples', 'y': 'Crop'},
    color=crop_counts.values,
    color_continuous_scale='Greens'
)

fig.update_layout(height=600)
fig.show()

In [ ]:
# Feature distributions by crop
features = ['N', 'P', 'K', 'temperature', 'humidity', 'ph', 'rainfall']

fig = make_subplots(
    rows=3, cols=3,
    subplot_titles=features
)

for i, feature in enumerate(features):
    row = i // 3 + 1
    col = i % 3 + 1
    
    for crop in df['crop'].unique()[:5]:  # Limit to 5 crops for clarity
        crop_data = df[df['crop'] == crop][feature]
        fig.add_trace(
            go.Histogram(x=crop_data, name=crop, opacity=0.7),
            row=row, col=col
        )

fig.update_layout(height=900, showlegend=True)
fig.show()

In [ ]:
# Correlation matrix
correlation = df[features].corr()

fig = px.imshow(
    correlation,
    text_auto=True,
    aspect="auto",
    color_continuous_scale='RdBu',
    title='Feature Correlation Matrix',
    labels=dict(color="Correlation")
)

fig.update_layout(height=600)
fig.show()

In [ ]:
# Box plots by crop
fig = make_subplots(
    rows=2, cols=4,
    subplot_titles=features
)

for i, feature in enumerate(features):
    row = i // 4 + 1
    col = i % 4 + 1
    
    for crop in df['crop'].unique():
        crop_data = df[df['crop'] == crop][feature]
        fig.add_trace(
            go.Box(y=crop_data, name=crop, showlegend=(i==0)),
            row=row, col=col
        )

fig.update_layout(height=800, showlegend=True)
fig.show()

In [ ]:
# Pairplot (sample for performance)
sample_df = df.sample(500)
g = sns.pairplot(sample_df, hue='crop', diag_kind='kde', height=2.5)
g.fig.suptitle('Feature Relationships by Crop', y=1.02, fontsize=16)
plt.tight_layout()
plt.show()

In [ ]:
# 3D Scatter plot
fig = px.scatter_3d(
    df.sample(500),
    x='N', y='P', z='K',
    color='crop',
    title='3D Nutrient Space by Crop',
    labels={'N': 'Nitrogen', 'P': 'Phosphorus', 'K': 'Potassium'}
)

fig.update_layout(height=700)
fig.show()

In [ ]:
# Radar chart for crop profiles
selected_crops = ['Rice', 'Wheat', 'Maize', 'Cotton']

fig = go.Figure()

for crop in selected_crops:
    crop_data = df[df['crop'] == crop][features].mean()
    
    fig.add_trace(go.Scatterpolar(
        r=crop_data.values,
        theta=features,
        fill='toself',
        name=crop
    ))

fig.update_layout(
    polar=dict(
        radialaxis=dict(
            visible=True,
            range=[0, df[features].max().max()]
        )),
    showlegend=True,
    title='Average Crop Profiles',
    height=600
)

fig.show()

In [ ]:
# Summary statistics by crop
summary = df.groupby('crop')[features].agg(['mean', 'std', 'min', 'max'])
summary

In [ ]:
# Key insights
print("🔍 KEY INSIGHTS")
print("="*50)
print(f"\n1. Dataset contains {len(df)} samples across {df['crop'].nunique()} crops")
print(f"\n2. Most common crop: {df['crop'].mode()[0]}")
print(f"\n3. Feature ranges:")
for feature in features:
    print(f"   - {feature}: {df[feature].min():.1f} to {df[feature].max():.1f}")
print(f"\n4. Strongest correlations:")
corr_matrix = df[features].corr()
for i in range(len(features)):
    for j in range(i+1, len(features)):
        if abs(corr_matrix.iloc[i, j]) > 0.5:
            print(f"   - {features[i]} vs {features[j]}: {corr_matrix.iloc[i, j]:.3f}")